# ???????????????

? notebook ?? `????????.md` ??????????? + ??? + ???????????

?????
1. ????????????? + ???? + ???????
2. ?????????????
3. ??????? Logit
4. ????????? GAS ???? Logit
5. ????????? B-spline ??? Logit???????
6. ???/????????
7. ??????AUC / KS / PSI + ???? + ????

???????????**???????**?????????? GAS/GAM ?????

## GitHub ??????????????????

???????? notebook ?????????????????????

- GAS ???????
  - https://github.com/vladimirholy/gasmodel
  - https://github.com/LeopoldoCatania/GAS
- ??/GAM ?????
  - https://github.com/dswah/pyGAM
  - https://github.com/statsmodels/statsmodels

?????
- ?????????????
- ???????????
- ?????????????

In [ ]:
# ===============================
# ?0???????????
# ???
# - SEED?????????????
# - EPS??????? log(0) / ????
# ===============================
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

SEED = 42
np.random.seed(SEED)
EPS = 1e-9


In [ ]:
# ===============================
# ?1????????
# ===============================
def sigmoid(x):
    """Logistic ?????????????????"""
    x = np.clip(x, -30, 30)
    return 1.0 / (1.0 + np.exp(-x))


def safe_auc(y_true, y_pred):
    """AUC ???????????????????? NaN?"""
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_pred))


def ks_score(y_true, y_pred):
    """KS ???TPR ? FPR ??????"""
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return np.nan
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    return float(np.max(np.abs(tpr - fpr)))


def psi_score(expected, actual, bins=10):
    """
    PSI?Population Stability Index??
    - expected????????????
    - actual????????/????
    """
    expected = np.asarray(expected)
    actual = np.asarray(actual)

    # ??????????
    cuts = np.quantile(expected, np.linspace(0, 1, bins + 1))
    cuts = np.unique(cuts)
    if len(cuts) < 3:
        return np.nan

    cuts[0], cuts[-1] = -np.inf, np.inf
    e = np.histogram(expected, bins=cuts)[0].astype(float)
    a = np.histogram(actual, bins=cuts)[0].astype(float)

    e = np.clip(e / e.sum(), 1e-6, None)
    a = np.clip(a / a.sum(), 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))


def psi_level(psi):
    """?????? PSI ?????/????/?????"""
    if pd.isna(psi):
        return "NA"
    if psi < 0.10:
        return "??"
    if psi <= 0.25:
        return "????"
    return "????"


def find_file(filename, root="."):
    """?????????????"""
    return next(Path(root).rglob(filename))


In [ ]:
# ===============================
# ?2????????
# ===============================
def load_borrower_data():
    """
    ????????
    - ???? x_ ???????
    - ???? user_id / y / ?? / ?? / ????
    - ????????????
    """
    path = find_file("model_sample_with_time_region.csv")
    df = pd.read_csv(path)

    x_cols = [c for c in df.columns if c.startswith("x_")]
    meta_cols = [c for c in df.columns if c not in x_cols]

    # ??????????
    cfg = {
        "id_col": meta_cols[0],
        "target_col": "y" if "y" in meta_cols else meta_cols[1],
        "time_col": meta_cols[2],
        "region_col": meta_cols[3],
        "province_col": meta_cols[4],
        "x_cols": x_cols,
    }

    df = df.rename(
        columns={
            cfg["id_col"]: "user_id",
            cfg["target_col"]: "y",
            cfg["time_col"]: "time",
            cfg["region_col"]: "region",
            cfg["province_col"]: "province",
        }
    )
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    return df, cfg


def load_govern_macro_data():
    """
    ?????????
    - ???????
    - ???????? gov_macro_XX?????????????????
    """
    path = find_file("govern_data.csv")
    df = pd.read_csv(path, encoding="utf-8")

    time_col = df.columns[0]
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.rename(columns={time_col: "time"})

    raw_macro_cols = [c for c in df.columns if c not in ["time", "province", "region"]]
    rename_map = {c: f"gov_macro_{i:02d}" for i, c in enumerate(raw_macro_cols, 1)}
    df = df.rename(columns=rename_map)
    macro_cols = list(rename_map.values())
    return df[["time", "province", "region"] + macro_cols], macro_cols


def quarter_label_to_date(text):
    """??2025?????????????????"""
    text = str(text)
    m = re.match(r"(\d{4})?\s*??([????1234])??", text)
    if not m:
        return pd.NaT

    year = int(m.group(1))
    q_map = {"?": 1, "?": 2, "?": 3, "?": 4, "1": 1, "2": 2, "3": 3, "4": 4}
    q = q_map[m.group(2)]
    month_day = {1: "03-31", 2: "06-30", 3: "09-30", 4: "12-31"}[q]
    return pd.Timestamp(f"{year}-{month_day}")


def load_national_macro_data_optional():
    """
    ???????????
    - ??????????????????
    - ????????????????
    """
    try:
        path = find_file("national_data.csv")
        df = pd.read_csv(path, encoding="gbk", skiprows=2)

        time_col = df.columns[0]
        df[time_col] = df[time_col].map(quarter_label_to_date)
        df = df.rename(columns={time_col: "time"})

        keep_cols = []
        for c in df.columns:
            if c == "time":
                continue
            s = pd.to_numeric(df[c], errors="coerce")
            if s.notna().sum() >= 8:
                df[c] = s
                keep_cols.append(c)

        rename_map = {c: f"nat_macro_{i:02d}" for i, c in enumerate(keep_cols, 1)}
        df = df.rename(columns=rename_map)
        nat_cols = list(rename_map.values())
        return df[["time"] + nat_cols], nat_cols
    except Exception:
        return pd.DataFrame(columns=["time"]), []


In [ ]:
# ===============================
# ?3???????????
# - GASDynamicLogit?Score-Driven ???? Logit
# - BSplineVaryingLogit?B-spline ??? Logit
# ===============================
class GASDynamicLogit:
    """
    ??? GAS ???? Logit???????????

        eta_{i,t} = x_{i,t}' beta + f_t
        f_t = omega + gamma' z_t + alpha * scaled_score_{t-1} + phi * f_{t-1}

    ???
    - z_t??? t ???????
    - score_{t-1}??? Bernoulli ???????????
    - scaled_score??? Fisher ????????
    """

    def __init__(self, l2=5e-3, maxiter=420):
        self.l2 = l2
        self.maxiter = maxiter

    @staticmethod
    def _split(theta, p, m):
        """??????? beta / gamma / omega / alpha / phi?"""
        beta = theta[:p]
        gamma = theta[p : p + m]
        omega = theta[p + m]
        alpha = theta[p + m + 1]
        phi = theta[p + m + 2]
        return beta, gamma, omega, alpha, phi

    def _forward(self, X, y_for_state, t_idx, z_by_t, theta):
        """
        ?????
        - ???? theta???????????
        - ??????????? f_t ? score_t???????
        """
        p = X.shape[1]
        m = z_by_t.shape[1]
        beta, gamma, omega, alpha, phi = self._split(theta, p, m)

        T = z_by_t.shape[0]
        out = np.zeros(X.shape[0])
        state = np.full(T, np.nan)
        score_arr = np.full(T, np.nan)

        # ???????? f_0?????????
        f_prev = (omega + z_by_t.mean(axis=0).dot(gamma)) / max(1.0 - phi, 1e-3)

        for t in range(T):
            mask = t_idx == t
            if t == 0:
                f_t = f_prev
                score_t = 0.0
            else:
                # score ?????????
                prev = t_idx == (t - 1)
                p_prev = sigmoid(X[prev].dot(beta) + f_prev)
                score_t = np.mean(y_for_state[prev] - p_prev)
                fisher_t = np.mean(p_prev * (1.0 - p_prev)) + EPS
                f_t = omega + z_by_t[t].dot(gamma) + alpha * (score_t / np.sqrt(fisher_t)) + phi * f_prev

            out[mask] = sigmoid(X[mask].dot(beta) + f_t)
            state[t] = f_t
            score_arr[t] = score_t
            f_prev = f_t

        return out, state, score_arr

    def _loss(self, theta, X, y, t_idx, z_by_t):
        """????? + L2 ???????????"""
        pred, _, _ = self._forward(X, y, t_idx, z_by_t, theta)
        nll = -np.sum(y * np.log(pred + EPS) + (1.0 - y) * np.log(1.0 - pred + EPS))

        p = X.shape[1]
        m = z_by_t.shape[1]
        beta, gamma, _, _, _ = self._split(theta, p, m)
        penalty = self.l2 * (np.sum(beta ** 2) + np.sum(gamma ** 2))
        return float(nll + penalty)

    def fit(self, X, y, t_idx, z_by_t):
        """
        ?????
        1) ???? Logit ? beta ??
        2) ?? L-BFGS-B ? MLE???????
        """
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        t_idx = np.asarray(t_idx, dtype=int)
        z_by_t = np.asarray(z_by_t, dtype=float)

        p = X.shape[1]
        m = z_by_t.shape[1]

        static = LogisticRegression(max_iter=300, solver="lbfgs")
        static.fit(X, y)
        beta0 = static.coef_.reshape(-1)

        y_bar = float(np.clip(y.mean(), 1e-4, 1 - 1e-4))
        omega0 = float(np.log(y_bar / (1 - y_bar)))
        theta0 = np.concatenate([beta0, np.zeros(m), np.array([omega0, 0.0, 0.2])])

        # ????????????????
        bounds = [(-4, 4)] * (p + m) + [(-6, 6), (-1, 1), (-0.95, 0.95)]

        res = minimize(
            self._loss,
            theta0,
            args=(X, y, t_idx, z_by_t),
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter": self.maxiter},
        )

        self.theta_ = res.x
        self.success_ = bool(res.success)
        self.message_ = str(res.message)
        self.p_ = p
        self.m_ = m
        return self

    def predict_proba(self, X, y_for_state, t_idx, z_by_t):
        """??????????????"""
        X = np.asarray(X, dtype=float)
        y_for_state = np.asarray(y_for_state, dtype=float)
        t_idx = np.asarray(t_idx, dtype=int)
        z_by_t = np.asarray(z_by_t, dtype=float)

        pred, _, _ = self._forward(X, y_for_state, t_idx, z_by_t, self.theta_)
        return pred

    def predict_proba_with_details(self, X, y_for_state, t_idx, z_by_t, time_labels):
        """???????????? GAS ????????????????"""
        X = np.asarray(X, dtype=float)
        y_for_state = np.asarray(y_for_state, dtype=float)
        t_idx = np.asarray(t_idx, dtype=int)
        z_by_t = np.asarray(z_by_t, dtype=float)

        pred, states, scores = self._forward(X, y_for_state, t_idx, z_by_t, self.theta_)
        detail = pd.DataFrame({
            "time": pd.to_datetime(time_labels),
            "gas_state": states,
            "gas_score": scores,
        })
        return pred, detail


class BSplineVaryingLogit:
    """
    ??? B-spline ??? Logit????????

        eta_i = x_i' beta + B(t_i)'theta0 + (x_key_i * B(t_i))'theta1

    ???
    - B(t_i)??????
    - theta0??????????
    - theta1????? x_key ?????
    """

    def __init__(self, n_knots=7, degree=3, lam=1.5, ridge=5e-3, maxiter=260):
        self.n_knots = n_knots
        self.degree = degree
        self.lam = lam
        self.ridge = ridge
        self.maxiter = maxiter

    def _build_design(self, X, t_num, key_feature, fit=False):
        """?????????[X, B(t), key*B(t)]?"""
        t_num = np.asarray(t_num).reshape(-1, 1)
        key_feature = np.asarray(key_feature)

        if fit:
            self.spline_ = SplineTransformer(
                n_knots=self.n_knots,
                degree=self.degree,
                include_bias=True,
                extrapolation="linear",
            )
            B = self.spline_.fit_transform(t_num)
        else:
            B = self.spline_.transform(t_num)

        K = B.shape[1]
        Xd = np.hstack([X, B, key_feature.reshape(-1, 1) * B])
        return Xd, K

    def _loss(self, w, Xd, y, p, K):
        """
        ?????
        - Bernoulli ?????
        - beta ? ridge ??
        - theta ??????????P-spline ???
        """
        pred = sigmoid(Xd.dot(w))
        nll = -np.sum(y * np.log(pred + EPS) + (1.0 - y) * np.log(1.0 - pred + EPS))

        beta = w[:p]
        theta0 = w[p : p + K]
        theta1 = w[p + K : p + 2 * K]

        if K >= 3:
            D2 = np.diff(np.eye(K), n=2, axis=0)
            smooth = np.sum((D2 @ theta0) ** 2) + np.sum((D2 @ theta1) ** 2)
        else:
            smooth = np.sum(theta0 ** 2) + np.sum(theta1 ** 2)

        return float(nll + self.ridge * np.sum(beta ** 2) + self.lam * smooth)

    def fit(self, X, y, t_num, key_feature):
        """?? L-BFGS-B ??????????????"""
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        Xd, K = self._build_design(X, t_num, key_feature, fit=True)
        p = X.shape[1]

        w0 = np.zeros(p + 2 * K)
        res = minimize(
            self._loss,
            w0,
            args=(Xd, y, p, K),
            method="L-BFGS-B",
            options={"maxiter": self.maxiter},
        )

        self.w_ = res.x
        self.success_ = bool(res.success)
        self.message_ = str(res.message)
        self.p_ = p
        self.K_ = K
        return self

    def predict_proba(self, X, t_num, key_feature):
        """?????????"""
        X = np.asarray(X, dtype=float)
        Xd, _ = self._build_design(X, t_num, key_feature, fit=False)
        return sigmoid(Xd.dot(self.w_))

    def get_time_effects(self, t_num_grid):
        """????????????????????"""
        B = self.spline_.transform(np.asarray(t_num_grid).reshape(-1, 1))
        p = self.p_
        K = self.K_
        theta0 = self.w_[p : p + K]
        theta1 = self.w_[p + K : p + 2 * K]
        return B.dot(theta0), B.dot(theta1)


In [ ]:
# ===============================
# ?4?????????
# ===============================
borrower_df, borrower_cfg = load_borrower_data()
govern_df, govern_cols = load_govern_macro_data()
national_df, national_cols = load_national_macro_data_optional()

# ?? time + province ????????????
model_df = borrower_df.merge(govern_df, on=["time", "province"], how="left")

# ??????????? time ??
if len(national_cols) > 0:
    model_df = model_df.merge(national_df, on="time", how="left")

# ????? region ????????????? region?
if "region" not in model_df.columns:
    if "region_x" in model_df.columns and "region_y" in model_df.columns:
        model_df["region"] = model_df["region_x"].fillna(model_df["region_y"])
    elif "region_x" in model_df.columns:
        model_df["region"] = model_df["region_x"]
    elif "region_y" in model_df.columns:
        model_df["region"] = model_df["region_y"]

for tmp_col in ["region_x", "region_y"]:
    if tmp_col in model_df.columns:
        model_df = model_df.drop(columns=tmp_col)

# ???????????? + ????
x_cols = borrower_cfg["x_cols"]
macro_cols = [c for c in model_df.columns if c.startswith("gov_macro_") or c.startswith("nat_macro_")]
candidate_cols = x_cols + macro_cols

# ??????
for c in candidate_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")

# ?????? / ??????
missing_ratio = model_df[candidate_cols].isna().mean()
keep_cols = [c for c in candidate_cols if missing_ratio[c] <= 0.98]
variance = model_df[keep_cols].var(numeric_only=True)
keep_cols = [c for c in keep_cols if variance.get(c, 0.0) > 1e-8]

print("??????:", borrower_df.shape)
print("???????:", model_df.shape)
print("?????:", len(candidate_cols), "?????:", len(keep_cols))
print("????:", model_df["time"].min(), "->", model_df["time"].max(), "???:", model_df["time"].nunique())
print("????:", model_df["region"].nunique(), "????:", model_df["province"].nunique())


In [ ]:
# ===============================
# ?5?????? + ?????
# ===============================
# ???????????????
all_times = np.sort(model_df["time"].dropna().unique())
train_times = all_times[:-6]
valid_times = all_times[-6:-3]
test_times = all_times[-3:]

train_mask = model_df["time"].isin(train_times)
valid_mask = model_df["time"].isin(valid_times)
test_mask = model_df["time"].isin(test_times)

# ??????????????????
corr_score = {}
for c in [k for k in keep_cols if k.startswith("x_")]:
    s = model_df.loc[train_mask, c]
    if s.notna().sum() < 200 or s.std(skipna=True) < 1e-12:
        corr_score[c] = 0.0
        continue
    cor = np.corrcoef(s.fillna(s.median()), model_df.loc[train_mask, "y"])[0, 1]
    corr_score[c] = abs(cor) if np.isfinite(cor) else 0.0

borrower_features = sorted(corr_score, key=corr_score.get, reverse=True)[:20]
macro_features = [c for c in keep_cols if c.startswith("gov_macro_")][:4]
selected_features = borrower_features + macro_features

# ????? + ???
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train = scaler.fit_transform(imputer.fit_transform(model_df.loc[train_mask, selected_features]))
X_valid = scaler.transform(imputer.transform(model_df.loc[valid_mask, selected_features]))
X_test = scaler.transform(imputer.transform(model_df.loc[test_mask, selected_features]))
X_all = scaler.transform(imputer.transform(model_df[selected_features]))

y_train = model_df.loc[train_mask, "y"].to_numpy()
y_valid = model_df.loc[valid_mask, "y"].to_numpy()
y_test = model_df.loc[test_mask, "y"].to_numpy()
y_all = model_df["y"].to_numpy()

print("???????:", len(selected_features))
print("??????Top8:", borrower_features[:8])
print("??????:", macro_features)
print("???? train/valid/test:", train_mask.sum(), valid_mask.sum(), test_mask.sum())


In [ ]:
# ===============================
# ?6???? GAS ??????????????
# ===============================
# ?????????????0,1,2,...?
time_order = np.sort(model_df["time"].dropna().unique())
time_to_idx = {t: i for i, t in enumerate(time_order)}
t_idx_all = model_df["time"].map(time_to_idx).to_numpy()

train_t_idx = np.array([time_to_idx[t] for t in train_times])

# ??????????? GAS ?????
state_cols = [c for c in macro_features if c in selected_features]
if len(state_cols) == 0:
    # ????????????3????????????
    state_cols = selected_features[:3]

# ??????????????
z_df = model_df.groupby("time")[state_cols].mean().sort_index()

# ????????? + ?????
z_df = z_df.ffill().bfill().fillna(z_df.median())

# ?????????????????
z_mu = z_df.iloc[train_t_idx].mean()
z_sd = z_df.iloc[train_t_idx].std().replace(0, 1.0)
z_std_df = (z_df - z_mu) / z_sd
z_by_t_all = z_std_df.to_numpy()

print("GAS ????:", state_cols)
print("??????:", z_by_t_all.shape)


In [ ]:
# ===============================
# ?7????????
# - ???? Logit
# - ??? GAS ?? Logit
# - ??? B-spline ??? Logit
# ===============================
# 1) ???? Logit
baseline = LogisticRegression(max_iter=400, solver="lbfgs")
baseline.fit(X_train, y_train)

# ??? train/valid/test ????????????
p_base = np.zeros(len(model_df))
p_base[train_mask] = baseline.predict_proba(X_train)[:, 1]
p_base[valid_mask] = baseline.predict_proba(X_valid)[:, 1]
p_base[test_mask] = baseline.predict_proba(X_test)[:, 1]

# 2) ??? GAS ?? Logit???????????
gas_model = GASDynamicLogit(l2=5e-3, maxiter=420)
gas_model.fit(
    X_train,
    y_train,
    t_idx_all[train_mask],
    z_by_t_all[train_t_idx],
)

# ???????? one-step ????????? y ?????
p_gas, gas_detail_df = gas_model.predict_proba_with_details(
    X_all,
    y_all,
    t_idx_all,
    z_by_t_all,
    time_order,
)

# 3) ??? B-spline ??? Logit
# ?????????????????????
key_feature = borrower_features[0] if len(borrower_features) > 0 else selected_features[0]
key_idx = selected_features.index(key_feature)

# ??????????????????
# ?? 2021Q3->0, 2021Q4->1, ...
t_num_all = np.array([time_to_idx[t] for t in model_df["time"]])

bs_model = BSplineVaryingLogit(n_knots=7, degree=3, lam=1.5, ridge=5e-3, maxiter=260)
bs_model.fit(
    X_train,
    y_train,
    t_num_all[train_mask],
    X_train[:, key_idx],
)
p_bs = bs_model.predict_proba(X_all, t_num_all, X_all[:, key_idx])

print("GAS ????:", gas_model.success_, gas_model.message_)
print("B-spline ????:", bs_model.success_, bs_model.message_)
print("B-spline ??????:", key_feature)


In [ ]:
# ===============================
# ?8?????????/????????
# ===============================
# ??????????????????
if z_by_t_all.shape[0] > 1:
    vol_series = np.r_[0.0, np.mean(np.abs(np.diff(z_by_t_all, axis=0)), axis=1)]
else:
    vol_series = np.zeros(z_by_t_all.shape[0])

vol_df = pd.DataFrame({"time": time_order, "volatility": vol_series})

# ??? 70% ????????
vol_threshold = float(vol_df.loc[vol_df["time"].isin(train_times), "volatility"].quantile(0.70))
vol_df["regime"] = np.where(vol_df["volatility"] >= vol_threshold, "volatile", "stable")

# ???????????
row_regime = model_df["time"].map(dict(zip(vol_df["time"], vol_df["regime"]))).to_numpy()

# ????????????p = w*GAS + (1-w)*B-spline
weight_grid = np.linspace(0.0, 1.0, 21)
best_w = {"stable": 0.5, "volatile": 0.5}

for regime in ["stable", "volatile"]:
    regime_mask = valid_mask.to_numpy() & (row_regime == regime)
    if regime_mask.sum() < 50 or np.unique(model_df.loc[regime_mask, "y"]).size < 2:
        continue

    best_auc = -1.0
    for w in weight_grid:
        p_try = w * p_gas[regime_mask] + (1.0 - w) * p_bs[regime_mask]
        auc_try = safe_auc(model_df.loc[regime_mask, "y"], p_try)
        if auc_try > best_auc:
            best_auc = auc_try
            best_w[regime] = float(w)

# ????????
weights = np.array([best_w[r] for r in row_regime])
p_ens = weights * p_gas + (1.0 - weights) * p_bs

print("????:", round(vol_threshold, 6))
print("????????GAS???:", best_w)


In [ ]:
# ===============================
# ?9???????AUC / KS / PSI?
# ===============================
def eval_split(y_true, y_pred):
    return {
        "auc": safe_auc(y_true, y_pred),
        "ks": ks_score(y_true, y_pred),
    }


records = []
for model_name, pred in [
    ("baseline", p_base),
    ("gas", p_gas),
    ("bspline", p_bs),
    ("ensemble", p_ens),
]:
    for split_name, split_mask in [
        ("train", train_mask),
        ("valid", valid_mask),
        ("test", test_mask),
    ]:
        y_true = model_df.loc[split_mask, "y"].to_numpy()
        y_hat = pred[split_mask]

        stat = eval_split(y_true, y_hat)
        psi = 0.0 if split_name == "train" else psi_score(pred[train_mask], pred[split_mask])

        records.append(
            {
                "model": model_name,
                "split": split_name,
                "n": int(split_mask.sum()),
                "default_rate": float(y_true.mean()),
                "auc": stat["auc"],
                "ks": stat["ks"],
                "psi_vs_train": psi,
                "psi_level": psi_level(psi),
            }
        )

# result_df ??????????????????
result_df = pd.DataFrame(records).sort_values(["model", "split"])
result_df


In [ ]:
# ===============================
# ?10?????????????
# ===============================
quarter_rows = []
for t, g in model_df.groupby("time"):
    y_t = g["y"].to_numpy()
    if np.unique(y_t).size < 2:
        continue

    p_base_t = p_base[g.index]
    p_gas_t = p_gas[g.index]
    p_bs_t = p_bs[g.index]
    p_ens_t = p_ens[g.index]

    quarter_rows.append(
        {
            "time": t,
            "n": len(g),
            "default_rate": float(y_t.mean()),
            "auc_base": safe_auc(y_t, p_base_t),
            "auc_gas": safe_auc(y_t, p_gas_t),
            "auc_bs": safe_auc(y_t, p_bs_t),
            "auc_ens": safe_auc(y_t, p_ens_t),
            "ks_ens": ks_score(y_t, p_ens_t),
        }
    )

quarter_eval_df = pd.DataFrame(quarter_rows).sort_values("time")

# ?? GAS ??????????????????????????
monitor_df = (
    quarter_eval_df.merge(gas_detail_df, on="time", how="left")
    .merge(vol_df, on="time", how="left")
)

monitor_df.tail(10)


In [ ]:
# ===============================
# ?11?????????????
# ===============================
region_rows = []
test_df = model_df.loc[test_mask].copy()

for region_name, g in test_df.groupby("region"):
    y_r = g["y"].to_numpy()
    if np.unique(y_r).size < 2:
        continue

    idx = g.index
    region_rows.append(
        {
            "region": region_name,
            "n": len(g),
            "default_rate": float(y_r.mean()),
            "auc_base": safe_auc(y_r, p_base[idx]),
            "auc_gas": safe_auc(y_r, p_gas[idx]),
            "auc_bs": safe_auc(y_r, p_bs[idx]),
            "auc_ens": safe_auc(y_r, p_ens[idx]),
            "ks_ens": ks_score(y_r, p_ens[idx]),
        }
    )

# region_eval_df ???????????????
region_eval_df = pd.DataFrame(region_rows).sort_values("auc_ens", ascending=False)
region_eval_df


In [ ]:
# ===============================
# ?12??B-spline ??????
# ===============================
# ???????????????? + ???????????
unique_t = np.arange(len(time_order))
intercept_curve, key_slope_curve = bs_model.get_time_effects(unique_t)

bs_effect_df = pd.DataFrame(
    {
        "time": pd.to_datetime(time_order),
        "bs_intercept_effect": intercept_curve,
        "bs_key_slope_effect": key_slope_curve,
    }
)

bs_effect_df.tail(10)


## ???????????????

??????????? `result_df / quarter_eval_df / region_eval_df / monitor_df` ???????????????


In [ ]:
# ===============================
# ?13????????????
# ===============================
# 1) ???????????
test_view = result_df[result_df["split"] == "test"].sort_values("auc", ascending=False).reset_index(drop=True)
best_model = test_view.loc[0, "model"]
best_auc = float(test_view.loc[0, "auc"])
best_ks = float(test_view.loc[0, "ks"])

print("???????????")
print(f"???AUC?????{best_model}?AUC={best_auc:.4f}?KS={best_ks:.4f}")
print("??????????AUC????")
print(test_view[["model", "auc", "ks", "psi_vs_train", "psi_level"]].to_string(index=False))

# 2) ??????PSI?
print()
print("?????????PSI??")
for _, row in test_view.iterrows():
    print(f"?? {row['model']}?PSI={row['psi_vs_train']:.4f}???={row['psi_level']}")
print("????????? PSI<0.1 ???0.1~0.25 ?????>0.25 ?????")

# 3) ????????????????
q = quarter_eval_df.copy().sort_values("time")
if len(q) > 0:
    worst_q = q.loc[q["auc_ens"].idxmin()]
    best_q = q.loc[q["auc_ens"].idxmax()]
    print()
    print("?????????????????")
    print(
        f"?????{best_q['time'].date()}?AUC={best_q['auc_ens']:.4f}?KS={best_q['ks_ens']:.4f}????={best_q['default_rate']:.4f}"
    )
    print(
        f"?????{worst_q['time'].date()}?AUC={worst_q['auc_ens']:.4f}?KS={worst_q['ks_ens']:.4f}????={worst_q['default_rate']:.4f}"
    )

# 4) ????
print()
print("?????????????")
if len(region_eval_df) > 0:
    top_region = region_eval_df.iloc[0]
    bottom_region = region_eval_df.iloc[-1]
    print(
        f"???????{top_region['region']}?AUC_ens={top_region['auc_ens']:.4f}?KS_ens={top_region['ks_ens']:.4f}"
    )
    print(
        f"???????{bottom_region['region']}?AUC_ens={bottom_region['auc_ens']:.4f}?KS_ens={bottom_region['ks_ens']:.4f}"
    )
    print("???????")
    print(region_eval_df.to_string(index=False))

# 5) ?????????????/????
print()
print("??????????")
if "regime" in monitor_df.columns:
    regime_mean = monitor_df.groupby("regime")[["auc_ens", "ks_ens", "default_rate"]].mean().reset_index()
    print("???????????????")
    print(regime_mean.to_string(index=False))
    print("?????????AUC/KS????????????????????????????????")

# 6) ??????????????
print()
print("???????????????")
print(
    f"???????{best_model}????????AUC={best_auc:.4f}?KS={best_ks:.4f}?"
    "?PSI????????????????????????????????????"
    "????????????????????????????????????"
    "????????? + ????? + ??????????????????"
)


## ???????????

1. ? notebook ??????????????????????????
2. ????????????`f_t`?`score_t`?`PSI`??????stable/volatile??????????
3. ????????/?????????????????????????????
4. ?????????????
   - GAS?`l2`?`maxiter`?`alpha/phi` ?????????
   - B-spline?`n_knots`?`lam`?????????
5. ?????????? notebook ??????? `model.py`??? `try.py` ????????